# Entity Extraction Approach

A rule-based entity extraction pipeline was developed to extract structured information from unstructured Indian bank transaction narrations. The pipeline identifies three key components from each narration:

- **Transaction Channel** – What payment channel or mode was used?
- **Counterparty** – Who was the actual merchant, person, or institution involved in the transaction?
- **Entity Role** – Was the counterparty sending or receiving money?

---

## Phase 1 – Text Normalization

Transaction narrations are first normalized to improve parsing consistency. Common spacing inconsistencies (e.g., `PH ONEPE`, `PHONE PE`, `PAY TM`, `GOOGLE PAY`) and noisy delimiters (such as pipe `|` characters) are standardized so that payment gateway detection remains reliable across different banks and narration formats.

---

## Phase 2 – Payment Channel Detection

The parser first determines whether a transaction is UPI-based. If so, it identifies the corresponding payment gateway whenever possible, including:

- PhonePe
- Paytm
- Google Pay
- Razorpay
- Airtel Payments
- BharatPe
- Amazon Pay
- SBIePay

If the transaction is not UPI, the parser classifies it into the appropriate banking channel, such as:

- IMPS
- NEFT
- RTGS
- ACH/NACH
- ATM
- Cash
- POS/Card
- Loan Recovery
- Bank Charges
- Interest
- Fund Transfer

---

## Phase 3 – Counterparty Extraction (UPI)

For UPI transactions, the parser extracts the most meaningful counterparty from the Virtual Payment Address (VPA). Typically, the portion before the `@` symbol represents the merchant or individual.

**Example:**

```
MYNTRA@ybl  →  Myntra
```

However, QR-generated handles such as:

```
paytmqr...@paytm
Q899838377@ybl
```

are not treated as valid counterparties because they represent QR identifiers rather than meaningful merchant or person names. These are classified as `UNRESOLVED_QR_MERCHANT`.

---

## Phase 4 – Counterparty Extraction (Non-UPI)

For non-UPI transactions—including IMPS, NEFT, IFN, ACH, POS/Card, ATM, Cash, and Loan Recovery narrations—the parser applies format-specific extraction rules to identify the most relevant merchant, institution, or individual mentioned in the narration. Each channel type has its own dedicated extractor that understands the narration structure specific to that payment mode.

---

## Phase 5 – UPI Handle-to-Bank Resolution

When the actual counterparty name cannot be extracted (e.g., due to masking or missing data), the parser resolves the UPI banking handle from the `@` suffix into a human-readable bank name using a comprehensive dictionary of 60+ Indian UPI handle mappings.

**Examples:**

| UPI Handle | Resolved Bank |
|---|---|
| `@ybl` | YES Bank |
| `@okhdfcbank` | HDFC Bank |
| `@ibl` | IndusInd Bank |
| `@paytm` | Paytm Payments Bank |
| `@ptyes` | YES Bank |
| `@superyes` | YES Bank |
| `@fdrl` | Federal Bank |
| `@cbin` | Central Bank of India |

The handle extraction regex is designed to stop at delimiter characters (dashes, dots), ensuring that routing metadata embedded after the handle (e.g., `@PTYES-BARB0KHARAG-689729445387-SENT`) does not contaminate the bank name resolution.

Coverage includes all major Indian banks (HDFC, ICICI, SBI, Axis, YES Bank, IndusInd, Kotak, PNB, Canara, Union, etc.), small finance and payments banks (Equitas, Ujjivan, AU, Fino, Jio, NSDL, India Post), and fintech wallets (Freecharge, MobiKwik, Slice, Jupiter, CRED).

---

## Phase 6 – Compound Word Splitting

Indian bank narrations frequently concatenate multiple tokens into a single word with no spaces—particularly when gateway names, bank names, and corporate suffixes are merged together. The parser applies a compound word splitting step before token-level noise removal, inserting spaces around recognized substrings so they can be individually identified and stripped.

**Examples:**

| Raw Concatenated Text | After Splitting | After Noise Removal |
|---|---|---|
| `Phonepeprivatel` | `Phonepe privatel` | _(both stripped → UNKNOWN)_ |
| `Playstoreaxisbank` | `Playstore axisbank` | _(both stripped → UNKNOWN)_ |
| `Jiocitibank` | `Jio citibank` | _(both stripped → UNKNOWN)_ |
| `Bhartiairtelltd` | `Bharti airtel ltd` | _(all stripped → UNKNOWN)_ |

This step ensures that gateway and bank names embedded inside compound words do not leak through as false counterparties.

---

## Phase 7 – Noise Removal and Edge-Case Handling

A comprehensive noise removal system operates at multiple levels:

### Token-Level Noise Removal

Over 120 noise tokens are stripped during extraction, organized into categories:

- **Transaction keywords:** `UPI`, `UPIOUT`, `PAYMENT`, `FROM`, `ACCOUNT`, `NA`, `SENT`, `PAID`, `ONLINE`
- **Payment modes:** `ACH`, `NACH`, `IMPS`, `NEFT`, `RTGS`
- **Corporate suffixes (including truncated forms):** `LIMITED`, `LTD`, `PRIVATE`, `PVT`, `LIMITE`, `LIMIT`, `PRIVATEL`
- **Bank names and acronyms:** `HDFC`, `ICICI`, `SBI`, `AXIS`, `YES`, `YESB`, `KOTAK`, `PNB`, `CANARA`, `INDUSIND`, and 40+ others
- **Payment gateways:** `PHONEPE`, `PAYTM`, `GOOGLEPAY`, `RAZORPAY`, `BHARATPE`, `AMAZONPAY`, `BILLDESK`, `CASHFREE`
- **Settlement and routing junk:** `ESCROW`, `PAYOUT`, `STLMNT`, `AGGR`, `AGG`, `SETTL`, `NOD`, `NODA`, `ACCO`, `MERCHANT`

### Pattern-Based Noise Removal

Regex patterns catch technical artifacts that cannot be enumerated as fixed tokens:

- **IFSC codes** (e.g., `HDFC0001234`)
- **Bank routing codes** (e.g., `UTIB00512`)
- **QR identifiers** (e.g., `PAYTMQR2847561`, `Q899838377`)
- **IMPS routing codes** (e.g., `F09`, `F10`)
- **Long alphanumeric reference IDs** (10+ mixed characters)
- **Masked mobile numbers** (e.g., `MOBILEXXXXX`)

### Edge-Case Classification

Instead of assigning all unresolved cases to a generic `UNKNOWN` category, the parser classifies common edge cases into more informative categories:

**Masked Privacy Strings**

Narrations containing heavily redacted identifiers (e.g., `MOBILEXXXX`) are classified with contextual metadata:

```
MASKED_COUNTERPARTY (Merchant) - HDFC Bank
MASKED_COUNTERPARTY (Individual) - YES Bank
MASKED_COUNTERPARTY - IndusInd Bank
```

The classification includes:
- **Entity type** when available: `(Merchant)` for P2M transactions, `(Individual)` for P2P/P2A transactions
- **Resolved bank name** from the UPI handle

**Unresolved QR Merchants**

QR-code-generated payment handles that cannot be traced to a specific merchant are classified as:

```
UNRESOLVED_QR_MERCHANT
```

**Transaction References**

Long mixed alphanumeric identifiers commonly found in ACH/NACH transactions or backend banking systems are classified as:

```
TRANSACTION_REFERENCE
```

**Unknown with Context**

When no counterparty can be extracted but transaction metadata is available, the parser provides context-enriched unknown labels:

```
UNKNOWN_MERCHANT (P2M) - Axis Bank
UNKNOWN_INDIVIDUAL (P2P) - HDFC Bank
```

**Gibberish Filtering**

Random character sequences with little linguistic meaning (e.g., `Zxzy`, `Zvm`) are detected by checking for the absence of vowels and rejected rather than being treated as valid counterparties.

---

## Phase 8 – Entity Role Classification

The counterparty's role is inferred from the transaction type:

| Transaction Type | Entity Role |
|---|---|
| CREDIT | Payer (Money Source) |
| DEBIT | Payee (Money Recipient) |

---

## Phase 9 – Audit Metadata

To improve transparency and facilitate manual review, the parser generates additional audit fields for every extraction, including:

- **Confidence Score** – Indicates the reliability of the extracted entity, ranging from 0.0 (no candidate found) to 0.99 (high-confidence masked entity detection). Different extraction methods carry different confidence levels:

| Confidence | Extraction Method |
|---|---|
| 0.99 | Masked entity detection |
| 0.86 | UPI VPA or name extraction |
| 0.88 | Loan recipient extraction |
| 0.82 | IMPS/IFN name extraction |
| 0.80 | NEFT named segment |
| 0.78 | ACH/Card merchant |
| 0.72 | Cash/ATM location |
| 0.64 | Best delimited segment |
| 0.60 | QR code detection |
| 0.45 | Fallback cleaned narration |
| 0.40 | Fallback P2M/P2P/reference |
| 0.00 | No candidate found |

- **Extraction Rule** – Records the specific rule or pattern responsible for the extraction (e.g., `upi_vpa_or_name`, `masked_mobile_rule`, `neft_named_segment`, `fallback_p2m`).

These audit fields make it easier to identify low-confidence extractions and review fallback cases separately.
# Pattern Detection & Feature Engineering Layer

A rule-based pattern detection layer that sits **after** the existing entity
extraction pipeline. Where entity extraction answers *"who was the
counterparty and what channel was used?"*, this layer answers *"what kind of
financial behavior does this transaction represent?"* — the signal actually
needed for credit underwriting and fraud review.

```
Narration
   ↓
Entity Extraction (Phases 1–9)   → PAYMENT_APP, COUNTERPARTY, ENTITY_ROLE
   ↓
Pattern Detection (Phase 10)     → TRANSACTION_PURPOSE, MERCHANT_CATEGORY, flags
   ↓
Feature Engineering (Phase 11)   → RECURRING_FLAG, salary stability, spend ratios
   ↓
Risk Variables                   → repayment capacity index, debt burden, etc.
```

---

## Phase 10 – Pattern Detection (row-level, narration-only)

Applied per transaction. Priority-ordered rule groups — first match wins,
so more specific categories are checked before generic fallbacks.

### Rule Priority (highest ROI for lending first)

1. **Income patterns** – Salary Credit, Bonus/Incentive, Interest, Dividend,
   Refund/Reversal, Cashback, Rental Income
2. **Loan / debt patterns** – Loan EMI, Loan Repayment, Loan Disbursement,
   Credit Card Bill Payment, BNPL/NBFC Repayment, Overdraft/Penalty
3. **Recurring bills / financial discipline** – Rent, Society Maintenance,
   Utility Bill, Telecom Recharge, Subscription/OTT/SaaS, Insurance Premium
4. **Investment patterns** – SIP/Mutual Fund, Direct Equity/Trading, Fixed
   Deposit, PF/NPS Contribution
5. **Government / statutory** – Tax Payment, Government Payment, Toll/FASTag
6. **Lifestyle / discretionary spend** – Food Delivery, E-commerce, Fuel,
   Ride-hailing, Travel Booking, Gaming/Betting, Healthcare, Education, Grocery
7. **Cash & transfer mechanics** – Cash Withdrawal, Cash Deposit, Wallet
   Loading, QR Payment, UPI Collect Request, Internal Account Transfer
8. **Business / vendor payment** – Invoice, Vendor, Purchase Order, GSTIN,
   Proprietorship/Firm keywords
9. **Fallback** – P2P Transfer or generic Merchant Payment, inferred from
   `COUNTERPARTY_TYPE` when no keyword rule fires

### Fields produced per row

| Field | Description |
|---|---|
| `TRANSACTION_PURPOSE` | Business-meaningful label (e.g. "Salary Credit", "Loan EMI") |
| `MERCHANT_CATEGORY` | Broad category bucket (Income, Debt Obligation, Investment, etc.) |
| `LIFESTYLE_CATEGORY` | Discretionary sub-category (Food & Dining, Shopping, Travel, etc.) or "N/A" |
| `COUNTERPARTY_TYPE` | Individual / Merchant / Business-Vendor / Financial Institution / Government / Unresolved |
| `DIGITAL_PAYMENT_FLAG` | True if payment mode isn't Cash/ATM/Unknown |
| `CASH_TRANSACTION_FLAG` | True for ATM/cash withdrawal or deposit narrations |
| `BUSINESS_PAYMENT_FLAG` | True if invoice/vendor/GSTIN-style keywords present |
| `FINANCIAL_INSTITUTION_FLAG` | True if narration references a bank/NBFC/insurer/AMC |
| `INCOME_FLAG` | True for credit-side income patterns |
| `LOAN_FLAG` | True for EMI/loan-repayment patterns |
| `EXPENSE_CATEGORY` | Debt Repayment / Fixed Expense / Savings-Investment / Discretionary Spend / Essential Spend / Other |
| `PATTERN_CONFIDENCE` | 0.0–0.85 depending on which rule tier matched |
| `PATTERN_RULE` | Name of the specific rule that fired (audit trail) |
| `RISK_TAG` | Cheap textual heuristic flag (e.g. "Discretionary Risk - Gambling", "Low Traceability") — a candidate flag for review, not a fraud score |

---

## Phase 11 – Feature Engineering (account-level, needs full statement)

These **cannot** be derived from a single narration; they need repetition
and, eventually, amount/date context across the account's transaction
history.

- **`RECURRING_FLAG`** — counterparty appears ≥ N times for the account
  (proxy for fixed obligations vs one-off spend)
- **`salary_stability_score`** — count of distinct "Salary Credit" hits
  (swap in monthly bucketing once a date column is available)
- **`discretionary_spend_ratio`** — share of debits classified as
  Discretionary Spend (transaction-count proxy until amount is wired in)

---

## Known Limitations

- Risk tags are narration-only heuristics. Real fraud/AML scoring needs
  amount + velocity layered on top.
- Income/recurring stability here is a transaction-count proxy. For a real
  repayment-capacity score, join in amount and statement date and re-bucket
  `salary_stability_score` and `discretionary_spend_ratio` by calendar month.
- Keyword dictionaries (merchant brand names, insurers, AMCs, NBFCs) will
  need periodic refresh as new fintech/merchant brands appear in statements.

In [2]:
import pandas as pd
from google.colab import files

uploaded = files.upload()

csv_file = list(uploaded.keys())[0]
df = pd.read_csv(csv_file)

df = df.dropna(subset=["NARRATION"]).copy()
df.head()

Saving Banking Sample Data for Entity Detection.csv to Banking Sample Data for Entity Detection.csv


,NARRATION,AAID,TYPE,MODE,AMOUNT,CURRENTBALANCE,VALUEDATE
0,UPIOUT/415442131843/Q899838377@ybl/Payment f/5411,aa91777914703824768192900488347968430422,DEBIT,OTHERS,70,7530,00:00.0
1,UPIOUT/415958720613/paytmqre4lk0uevrg@paytm//5451,aa91777914703824768192900488347968430422,DEBIT,OTHERS,60,1649,00:00.0
2,UPIOUT/416160374009/paytmqr14clec@paytm/Paym/5812,aa91777914703824768192900488347968430422,DEBIT,OTHERS,36,8730,00:00.0
3,UPIOUT/415838737884/Q047024853@ybl/Payment f/5812,aa91777914703824768192900488347968430422,DEBIT,OTHERS,5,8019,00:00.0
4,UPIOUT/416102104755/paytmqr281005050101wntva/5411,aa91777914703824768192900488347968430422,DEBIT,OTHERS,24,11040,00:00.0


In [17]:
from __future__ import annotations

import re
from dataclasses import asdict, dataclass
from functools import lru_cache
from typing import Mapping

# ═════════════════════════════════════════════════════════════════════════════
# Constants
# ═════════════════════════════════════════════════════════════════════════════

UNKNOWN = "UNKNOWN"
MASKED_COUNTERPARTY = "MASKED_COUNTERPARTY"
TRANSACTION_REFERENCE = "TRANSACTION_REFERENCE"


@dataclass(frozen=True)
class ExtractionResult:
    narration: str
    payment_mode: str
    counterparty: str
    flow: str
    confidence: float
    rule: str

    def to_dict(self) -> dict[str, str | float]:
        return asdict(self)


# ═════════════════════════════════════════════════════════════════════════════
# Phase 2 – Payment Channel Detection
# ═════════════════════════════════════════════════════════════════════════════

GATEWAY_PATTERNS: tuple[tuple[str, re.Pattern[str]], ...] = (
    ("PhonePe",         re.compile(r"\bPH\s*ONE\s*PE\b|\bPHONE\s*PE\b|\bPHONEPE\b|@YBL\b|@AXL\b", re.I)),
    ("Paytm",           re.compile(r"\bPAY\s*TM\b|\bPAYTM\b|@PAYTM\b|PAYTMQR", re.I)),
    ("Google Pay",      re.compile(r"\bGOOGLE\s*PAY\b|\bGPAY\b|@OK(?:AXIS|AXI|HDFC|ICICI|SBI)\b", re.I)),
    ("Razorpay",        re.compile(r"\bRAZOR\s*PAY\b|\bRAZORPAY\b|\.RZP@|@RZP|RZP@", re.I)),
    ("Airtel Payments", re.compile(r"\bAIRTEL\b|@MAIRTEL\b|RXAIRTEL", re.I)),
    ("BharatPe",        re.compile(r"\bBHARAT\s*PE\b|\bBHARATPE\b", re.I)),
    ("Amazon Pay",      re.compile(r"\bAMAZON\s*PAY\b|\bAMAZONPAY\b", re.I)),
    ("SBIePay",         re.compile(r"\bSBI\s*E\s*PAY\b|\bSBIEPAY\b", re.I)),
)

MODE_PATTERNS: tuple[tuple[str, re.Pattern[str]], ...] = (
    ("ATM",           re.compile(r"\b(?:ATM|NFS|ATW)\b|ATM CASH|CASH TXN", re.I)),
    ("Cash",          re.compile(r"\bCASH\b|CASH DEPOSIT|CASH WDL", re.I)),
    ("IMPS",          re.compile(r"\bIMPS\b|\.IMPS", re.I)),
    ("NEFT",          re.compile(r"\bNEFT\b", re.I)),
    ("RTGS",          re.compile(r"\bRTGS\b", re.I)),
    ("ACH/NACH",      re.compile(r"\b(?:ACH|ACHCR|NACH|ECS)\b", re.I)),
    ("POS/Card",      re.compile(r"\b(?:POS|PRCR|CARD|VISA|MASTER(?:CARD)?)\b", re.I)),
    ("UPI",           re.compile(r"\bUPI(?:OUT|AB|AR)?\b|UPI\s+IN|@[A-Z0-9._-]+", re.I)),
    ("Loan Recovery", re.compile(r"\bLOAN\s+REC\b|\bLOAN\b", re.I)),
    ("Charges",       re.compile(r"\bCHARGE\b|\bCHARGES\b|\bCHG\b", re.I)),
    ("Interest",      re.compile(r"\bINTEREST\b|INT\.? CREDIT", re.I)),
    ("Fund Transfer", re.compile(r"\b(?:FT|TRTR|MB|MBK|INET|INF|IFN|MMT|RECD)\b", re.I)),
)

# OPTIMIZATION: this exact regex was previously written out twice — once inline
# inside _is_upi() and once inline inside _payment_mode(). Both now share one
# compiled pattern, so it's compiled once and there's a single source of truth.
_UPI_DETECT_RE = re.compile(r"\bUPI(?:OUT|AB|AR)?\b|UPI\s+IN|@[A-Z0-9._-]+", re.I)

# OPTIMIZATION: P2M/P2P/P2A checks were repeated as inline re.search() calls in
# three separate places in _counterparty(). Compiling them once avoids
# recompiling the same small pattern on every row.
_P2M_RE = re.compile(r"\bP2M\b", re.I)
_P2P_RE = re.compile(r"\bP2P\b|\bP2A\b", re.I)


# ═════════════════════════════════════════════════════════════════════════════
# Phase 5 – UPI Handle-to-Bank Resolution
# ═════════════════════════════════════════════════════════════════════════════

UPI_HANDLE_BANK_MAP = {
    "okhdfcbank": "HDFC Bank", "hdfcbank": "HDFC Bank", "hdfc": "HDFC Bank",
    "okicici": "ICICI Bank", "icici": "ICICI Bank",
    "oksbi": "State Bank of India", "sbi": "State Bank of India", "sbin": "State Bank of India",
    "okaxis": "Axis Bank", "axisbank": "Axis Bank", "axl": "Axis Bank", "utib": "Axis Bank",
    "ybl": "YES Bank", "yesbank": "YES Bank", "yesbankltd": "YES Bank",
    "yes": "YES Bank", "superyes": "YES Bank", "ptyes": "YES Bank",
    "yescred": "YES Bank", "yespop": "YES Bank", "yesc": "YES Bank",
    "nyes": "YES Bank", "yesba": "YES Bank", "yespay": "YES Bank",
    "ibl": "IndusInd Bank", "indus": "IndusInd Bank",
    "kotak": "Kotak Mahindra Bank", "kkbk": "Kotak Mahindra Bank",
    "paytm": "Paytm Payments Bank",
    "apl": "Airtel Payments Bank", "apb": "Airtel Payments Bank",
    "mairtel": "Airtel Payments Bank", "airtel": "Airtel Payments Bank", "airp": "Airtel Payments Bank",
    "idfc": "IDFC First Bank", "idfcbank": "IDFC First Bank",
    "fbl": "Federal Bank", "fdrl": "Federal Bank",
    "barodampay": "Bank of Baroda", "barb": "Bank of Baroda", "bob": "Bank of Baroda",
    "rbl": "RBL Bank", "ratn": "RBL Bank",
    "cbin": "Central Bank of India",
    "ubi": "Union Bank of India", "ubin": "Union Bank of India", "unionbank": "Union Bank of India",
    "pnb": "Punjab National Bank", "punb": "Punjab National Bank",
    "cnrb": "Canara Bank", "canara": "Canara Bank",
    "indianbk": "Indian Bank", "idib": "Indian Bank",
    "bkid": "Bank of India", "boi": "Bank of India",
    "ioba": "Indian Overseas Bank",
    "karb": "Karnataka Bank",
    "sibl": "South Indian Bank",
    "bandhan": "Bandhan Bank", "bdbl": "Bandhan Bank",
    "idbi": "IDBI Bank", "ibkl": "IDBI Bank",
    "ucba": "UCO Bank", "uco": "UCO Bank",
    "mahb": "Bank of Maharashtra",
    "psib": "Punjab & Sind Bank",
    "citi": "Citibank", "citibank": "Citibank",
    "equitas": "Equitas Small Finance Bank",
    "ujjivan": "Ujjivan Small Finance Bank",
    "aubank": "AU Small Finance Bank",
    "fino": "Fino Payments Bank",
    "jio": "Jio Payments Bank",
    "nsdl": "NSDL Payments Bank",
    "ipos": "India Post Payments Bank",
    "freecharge": "Freecharge",
    "mobikwik": "MobiKwik",
    "slice": "Slice",
    "jupiter": "Jupiter",
    "cred": "CRED",
    "sibi": "State Bank of India (Intl)",
}

_HANDLE_RE = re.compile(r"@([A-Za-z0-9]+)", re.I)


def _resolve_bank_from_handle(raw_text: str) -> str:
    """Phase 5 – UPI Handle-to-Bank Resolution."""
    handle_match = _HANDLE_RE.search(raw_text)
    if not handle_match:
        return ""
    raw_handle = handle_match.group(1).lower()
    return UPI_HANDLE_BANK_MAP.get(raw_handle, f"@{raw_handle}")


# ═════════════════════════════════════════════════════════════════════════════
# Phase 7 – Noise Removal
# ═════════════════════════════════════════════════════════════════════════════

NOISE_WORDS = {
    "", "PAY", "PAYM", "PAYMENT", "PAYMENTS", "PAYVIA", "FROM", "FOR", "REF",
    "TRANSFER", "FUND", "FUNDS", "IFN", "IN", "PRCR", "UPI", "UPIOUT", "UPIIN",
    "UPIAR", "UPIAB", "CR", "DR", "DEBIT", "CREDIT", "OUTWARD", "INWARD",
    "ACCOUNT", "MOBILEXXXX", "MOBILEXXXXX", "NA", "SENT", "PAID", "RECEIVED",
    "TXN", "TRANSACTION", "ONLINE", "PA", "SO", "TO", "BY", "VIA", "THE",
    "ACH", "NACH", "ECS", "IMPS", "NEFT", "RTGS", "FT", "POS",
    "LIMITED", "LTD", "PRIVATE", "PVT", "BANK", "BANKING",
    "FINANCE", "FINSERV", "FINTECH",
    "TECHNOLOGIES", "TECH", "SOLUTIONS", "SERVICES",
    "INDIA", "CORP", "CORPORATION",
    "LIMITE", "LIMIT", "PRIVATEL", "PRIVAT",
    "DIGITAL", "COMMUNICATIONS", "COMM",
    "ENTERPRISES", "VENTURES", "ASSOCIATES",
    "HDFC", "HDFCBANK",
    "ICICI", "ICICIBANK",
    "SBI", "SBIN", "STATE",
    "AXIS", "AXISBANK", "UTIB",
    "KOTAK", "KKBK",
    "YES", "YESB", "YESBANK", "YBS", "YESPAY",
    "INDUSIND", "INDUS", "IBL",
    "PNB", "PUNB", "PUNJAB", "NATIONAL",
    "BOB", "BARB", "BARODA",
    "IDFC", "IDFCBANK",
    "FEDERAL", "FDRL", "FBL",
    "RBL", "RATN",
    "CANARA", "CNRB",
    "UNION", "UBI", "UBIN",
    "CENTRAL", "CBIN",
    "INDIAN", "INDIANBK", "IDIB",
    "BOI", "BKID",
    "IOB", "IOBA",
    "KARB", "KARNATAKA",
    "SIBL",
    "BANDHAN", "BDBL",
    "IDBI", "IBKL",
    "UCO", "UCBA",
    "MAHB", "MAHARASHTRA",
    "PSIB",
    "EQUITAS", "UJJIVAN", "AUBANK",
    "FINO", "JIO", "NSDL",
    "CITI", "CITIBANK",
    "IPOS",
    "PHONEPE", "PAYTM", "GOOGLEPAY", "GPAY", "GOOGLE",
    "RAZORPAY", "BHARATPE", "AMAZONPAY", "RZP",
    "BILLDESK", "CCAVENUE", "PAYU", "CASHFREE", "PINELABS", "DIGIPE",
    "BHARTI", "AIRTEL", "BHARTIAIRTEL",
    "PLAYSTORE",
    "ESCROW", "PAYOUT", "PAYOUTS", "STLMNT", "DOTRANSACTION",
    "AGGR", "AGG", "AGGREGATOR",
    "SETTL", "SETTLEMENT", "SETTLE",
    "NOD", "NODA", "NODAL",
    "ACCO", "MERCHANT",
    "RAPIPAY",
}


# ═════════════════════════════════════════════════════════════════════════════
# Phase 6 – Compound Word Splitting
# ═════════════════════════════════════════════════════════════════════════════

_COMPOUND_GATEWAY_RE = re.compile(
    r"(phonepe|paytm|razorpay|bharatpe|amazonpay|googlepay|gpay"
    r"|sbiepay|billdesk|cashfree|pinelabs|playstore"
    r"|bhartiairtel|bharti|airtel|google|rapipay|digipe)",
    re.I,
)

_COMPOUND_BANK_RE = re.compile(
    r"(axisbank|hdfcbank|icicibank|indusind|citibank|yesbank"
    r"|kotakbank|canarabank|federalbank|idfcbank)",
    re.I,
)

_COMPOUND_SUFFIX_RE = re.compile(
    r"(privatel|private|limited|ltd|limite|limit|pvt)(?=[a-z]|$)",
    re.I,
)


# ═════════════════════════════════════════════════════════════════════════════
# Phase 7 (continued) – Pattern-Based Noise Removal
#
# OPTIMIZATION: previously stored as a tuple of 12 separately compiled
# patterns, and checked with `any(pattern.match(upper) for pattern in ...)`
# — meaning up to 12 regex engine invocations per token, for every token,
# in every row. All 12 are now folded into ONE compiled alternation, so
# each token/candidate needs exactly one regex match instead of up to 12.
# The individual patterns are kept below (unused directly) purely as
# documentation of what each branch of the combined pattern represents.
# ═════════════════════════════════════════════════════════════════════════════

BANK_OR_TECH_PATTERNS = (
    re.compile(r"^[A-Z]{4}0[A-Z0-9]{6}$"),           # IFSC codes (e.g., HDFC0001234)
    re.compile(r"^[A-Z]{3,6}\d{3,}$"),                # Bank routing codes (e.g., UTIB00512)
    re.compile(r"^PAYTMQR[A-Z0-9]*$", re.I),          # Paytm QR IDs
    re.compile(r"^Q\d+[A-Z0-9]*$", re.I),             # Generic QR IDs (e.g., Q899838377)
    re.compile(r"^[A-Z]{1,2}\d{5,}[A-Z0-9]*$", re.I), # QR/Reference prefixes (e.g., Zq7206840)
    re.compile(r"^P2[APM]$", re.I),                    # UPI transaction type flags
    re.compile(r"^\d{6,}$"),                           # Long numeric strings
    re.compile(r"^X{2,}\d*$", re.I),                   # Masked digits (XX, XXX...)
    re.compile(r"^XXXXX\d*$", re.I),                   # Masked account numbers
    re.compile(r"^MOBILEX+[A-Z0-9]*$", re.I),          # Masked mobile numbers
    re.compile(r"^(?=.*[A-Z])(?=.*\d)[A-Z0-9]{10,}$", re.I),  # Long alphanumeric reference IDs
    re.compile(r"^F\d{2}$", re.I),                     # IMPS routing codes (F09, F10, etc.)
)

_BANK_OR_TECH_COMBINED_RE = re.compile(
    "|".join(f"(?:{p.pattern})" for p in BANK_OR_TECH_PATTERNS), re.I
)


def _is_bank_or_tech(value: str) -> bool:
    """Single-pass replacement for `any(p.match(value) for p in BANK_OR_TECH_PATTERNS)`."""
    return bool(_BANK_OR_TECH_COMBINED_RE.match(value))


# ═════════════════════════════════════════════════════════════════════════════
# Phase 7 (continued) – Standalone Bank Handle Stripping
# ═════════════════════════════════════════════════════════════════════════════

_STANDALONE_HANDLE_RE = re.compile(
    r"\b(?:OKHDFC(?:BANK)?|OKICICI|OKSBI|OKAXIS"
    r"|YBL|IBL|AXL"
    r"|YESB|YESBANKLTD|YESBANK|SUPERYES|PTYES|YESCRED|YESPOP|YESPAY"
    r"|HDFCBANK|AXISBANK|IDFCBANK"
    r"|PHONEPEMERCHANT)\b",
    re.I,
)


# ═════════════════════════════════════════════════════════════════════════════
# Core Pipeline
# ═════════════════════════════════════════════════════════════════════════════

def parse_row(row: Mapping[str, str]) -> ExtractionResult:
    return parse_narration(
        row.get("NARRATION", ""),
        transaction_type=row.get("TYPE", ""),
    )


def parse_narration(narration: str, transaction_type: str = "") -> ExtractionResult:
    raw = narration or ""
    normalized = _normalize(raw)
    flow = _flow_from_type(transaction_type)
    payment_mode = _payment_mode(normalized)
    counterparty, confidence, rule = _counterparty(normalized, raw)

    return ExtractionResult(
        narration=raw,
        payment_mode=payment_mode,
        counterparty=counterparty,
        flow=flow,
        confidence=confidence,
        rule=rule,
    )


def _flow_from_type(transaction_type: str) -> str:
    tx_type = (transaction_type or "").strip().upper()
    if tx_type == "DEBIT":
        return "Receiver"
    if tx_type == "CREDIT":
        return "Sender"
    return UNKNOWN


def _payment_mode(text: str) -> str:
    # OPTIMIZATION: reuses the shared _UPI_DETECT_RE constant instead of an
    # inline regex literal (was previously identical to _is_upi's pattern,
    # written out separately).
    if _UPI_DETECT_RE.search(text):
        for gateway, pattern in GATEWAY_PATTERNS:
            if pattern.search(text):
                return gateway
        return "UPI"

    for mode, pattern in MODE_PATTERNS:
        if pattern.search(text):
            return mode
    return UNKNOWN


def _counterparty(text: str, raw: str) -> tuple[str, float, str]:

    # ── Phase 7 – Masked Privacy Strings ─────────────────────────────────
    if _is_masked_mobile(raw):
        suffix = ""
        if _P2M_RE.search(raw):
            suffix = " (Merchant)"
        elif _P2P_RE.search(raw):
            suffix = " (Individual)"

        bank_name = _resolve_bank_from_handle(raw)
        if bank_name:
            suffix += f" - {bank_name}"

        return f"MASKED_COUNTERPARTY{suffix}", 0.99, "masked_mobile_rule"

    # ── Phase 3 – Counterparty Extraction (UPI) ─────────────────────────
    if _is_upi(text):
        name = _extract_upi_counterparty(text)
        if name != UNKNOWN:
            return name, 0.86, "upi_vpa_or_name"

    # ── Phase 4 – Counterparty Extraction (Non-UPI) ─────────────────────
    # OPTIMIZATION: _parts(text) is now cached (see below), so even though
    # each of these 8 extractors calls _parts(text) independently, the
    # expensive re.split() work happens only once per distinct narration
    # instead of up to 8 times.
    for extractor, confidence, rule in (
        (_extract_loan_counterparty,      0.88, "loan_recipient"),
        (_extract_ifn_counterparty,       0.82, "ifn_tail_name"),
        (_extract_imps_recd_counterparty, 0.82, "imps_recd_name"),
        (_extract_neft_counterparty,      0.80, "neft_named_segment"),
        (_extract_ach_counterparty,       0.78, "ach_originator"),
        (_extract_card_counterparty,      0.78, "card_merchant"),
        (_extract_cash_atm_counterparty,  0.72, "cash_atm_location"),
        (_extract_delimited_counterparty, 0.64, "best_delimited_segment"),
    ):
        candidate = extractor(text)
        if candidate != UNKNOWN:
            return candidate, confidence, rule

    # ── Phase 7 – Fallback: clean the raw narration ─────────────────────
    cleaned = _clean_name(raw)
    if cleaned and not _is_gibberish(cleaned):
        return cleaned, 0.45, "fallback_cleaned_narration"

    # ── Phase 7 – Informative Unknown Labels ────────────────────────────
    if re.search(r"PAYTMQR|^(?:QR|Q)\d{5,}", raw, re.I):
        return "UNRESOLVED_QR_MERCHANT", 0.60, "qr_code_detected"

    for part in _parts(text):
        if _is_reference_like(part):
            return TRANSACTION_REFERENCE, 0.40, "fallback_reference_id"

    bank_name = _resolve_bank_from_handle(raw)
    bank_suffix = f" - {bank_name}" if bank_name else ""

    if _P2M_RE.search(raw):
        return f"UNKNOWN_MERCHANT (P2M){bank_suffix}", 0.40, "fallback_p2m"
    if _P2P_RE.search(raw):
        return f"UNKNOWN_INDIVIDUAL (P2P){bank_suffix}", 0.40, "fallback_p2p"

    return UNKNOWN, 0.0, "no_candidate"


# ═════════════════════════════════════════════════════════════════════════════
# Phase 1 – Text Normalization
# ═════════════════════════════════════════════════════════════════════════════

def _normalize(text: str) -> str:
    text = text.replace("\ufeff", " ")
    text = re.sub(r"PH\s+ONE\s*PE|PHONE\s+PE", "PHONEPE", text, flags=re.I)
    text = re.sub(r"PAY\s+TM", "PAYTM", text, flags=re.I)
    text = re.sub(r"GOOGLE\s+PAY", "GOOGLEPAY", text, flags=re.I)
    text = re.sub(r"[|]+", "/", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def _is_upi(text: str) -> bool:
    # OPTIMIZATION: shares _UPI_DETECT_RE with _payment_mode instead of
    # re-declaring the same regex literal.
    return bool(_UPI_DETECT_RE.search(text))


# ═════════════════════════════════════════════════════════════════════════════
# Phase 3 & 4 – Channel-Specific Counterparty Extractors
# ═════════════════════════════════════════════════════════════════════════════

def _extract_upi_counterparty(text: str) -> str:
    """Phase 3 – UPI: Extract counterparty from VPA (portion before @)."""
    parts = _parts(text)
    vpa_parts = [p for p in parts if "@" in p and not _is_masked_mobile(p)]

    for part in vpa_parts:
        before_at = part.split("@", 1)[0]
        candidate = _clean_vpa_handle(before_at)
        if _is_good_name(candidate):
            return candidate

    for part in parts:
        candidate = _clean_name(part)
        if _is_good_name(candidate):
            return candidate
    return UNKNOWN


def _extract_loan_counterparty(text: str) -> str:
    """Phase 4 – Loan Recovery: Extract recipient from LOAN REC narrations."""
    match = re.search(r"LOAN\s+REC[:/\s-]*(?:\d+\s*)?(?P<name>.+)$", text, re.I)
    return _clean_name(match.group("name")) if match else UNKNOWN


def _extract_ifn_counterparty(text: str) -> str:
    """Phase 4 – IFN: Extract name from the tail of IFN/ narrations."""
    match = re.search(r"\bIFN/[^ ]+\s+(?P<name>.+)$", text, re.I)
    return _clean_name(match.group("name")) if match else UNKNOWN


def _extract_imps_recd_counterparty(text: str) -> str:
    """Phase 4 – IMPS/RECD/MMT: Extract name from delimited segments."""
    parts = _parts(text)
    for marker in ("RECD", "IMPS", "MMT"):
        if parts and parts[0].upper().startswith(marker):
            for part in parts[2:]:
                candidate = _clean_name(part)
                if _is_good_name(candidate):
                    return candidate
    return UNKNOWN


def _extract_neft_counterparty(text: str) -> str:
    """Phase 4 – NEFT: Extract name from reversed segment scan."""
    parts = _parts(text)
    if not any(p.upper().startswith(("NEFT", "N")) for p in parts[:2]):
        return UNKNOWN
    for part in reversed(parts):
        candidate = _clean_name(part)
        if _is_good_name(candidate):
            return candidate
    return UNKNOWN


def _extract_ach_counterparty(text: str) -> str:
    """Phase 4 – ACH/NACH/ECS: Extract originator name."""
    parts = _parts(text)
    if not parts or not re.search(r"ACH|NACH|ECS", parts[0], re.I):
        return UNKNOWN
    for part in parts[1:]:
        candidate = _clean_name(part)
        if _is_good_name(candidate):
            return candidate
    return UNKNOWN


def _extract_card_counterparty(text: str) -> str:
    """Phase 4 – POS/Card: Extract merchant name."""
    parts = _parts(text)
    if not re.search(r"\b(?:POS|PRCR|CARD)\b", text, re.I):
        return UNKNOWN
    for part in parts[1:]:
        candidate = _clean_name(part)
        if _is_good_name(candidate):
            return candidate
    return UNKNOWN


def _extract_cash_atm_counterparty(text: str) -> str:
    """Phase 4 – ATM/Cash: Extract location or branch name."""
    if not re.search(r"\b(?:ATM|NFS|ATW|CASH)\b", text, re.I):
        return UNKNOWN
    parts = _parts(text)
    for part in reversed(parts):
        candidate = _clean_name(part)
        if _is_good_name(candidate):
            return candidate
    return UNKNOWN


def _extract_delimited_counterparty(text: str) -> str:
    """Phase 4 – Generic: Extract best candidate from delimited segments."""
    for part in _parts(text):
        candidate = _clean_name(part)
        if _is_good_name(candidate):
            return candidate
    return UNKNOWN


@lru_cache(maxsize=8192)
def _parts(text: str) -> tuple[str, ...]:
    """
    Split narration text on / and # delimiters into segments.

    OPTIMIZATION: memoized with lru_cache. In the original code this function
    was called independently by _extract_upi_counterparty and by up to 8
    non-UPI extractors for the SAME narration string — recomputing the same
    re.split() result every time. Since narrations repeat heavily across a
    bank statement (same merchants recur constantly), caching by the input
    string turns most of those repeat calls into O(1) dict lookups. Returns
    a tuple instead of a list so the cached result is safely immutable and
    shareable across calls (all existing usages — indexing, slicing,
    `reversed()`, iteration — work identically on a tuple).
    """
    return tuple(part.strip() for part in re.split(r"[/#]", text) if part.strip())


# ═════════════════════════════════════════════════════════════════════════════
# Phases 6 & 7 – Name Cleaning & Validation
# ═════════════════════════════════════════════════════════════════════════════

def _clean_vpa_handle(handle: str) -> str:
    """Phase 3 helper – Cleans a UPI VPA handle (the portion before @)."""
    compact = re.sub(r"[^A-Za-z0-9]", "", handle)
    if re.match(r"^PAYTMQR|^(?:QR|Q)\d", compact, re.I):
        return UNKNOWN
    if _digit_ratio(compact) > 0.45:
        return UNKNOWN
    handle = re.sub(r"\bMOBILEX+\b", "", handle, flags=re.I)
    handle = re.sub(r"^(?:paytmqr|qr|q)(?=[a-z0-9])", "", handle, flags=re.I)
    handle = re.sub(r"[-_.]+", " ", handle)
    handle = re.sub(r"\d+", " ", handle)

    handle = _COMPOUND_GATEWAY_RE.sub(r" \1 ", handle)
    handle = _COMPOUND_BANK_RE.sub(r" \1 ", handle)
    handle = _COMPOUND_SUFFIX_RE.sub(r" \1 ", handle)

    return _title(_strip_noise(handle))


def _clean_name(value: str) -> str:
    """Master cleaning function for counterparty name candidates."""
    value = re.sub(r"@[A-Z0-9._-]+", " ", value, flags=re.I)
    value = re.sub(r"\bMOBILEX+\b", " ", value, flags=re.I)
    value = re.sub(r"\bX{4,}\d*\b", " ", value, flags=re.I)
    value = re.sub(r"\b\d{2}[-/]\d{2}[-/]\d{2,4}\b", " ", value)
    value = re.sub(r"\b\d{1,2}:\d{2}(?::\d{2})?\b", " ", value)
    value = re.sub(r"\b\d{4,}\b", " ", value)

    value = _STANDALONE_HANDLE_RE.sub(" ", value)

    value = _COMPOUND_GATEWAY_RE.sub(r" \1 ", value)
    value = _COMPOUND_BANK_RE.sub(r" \1 ", value)
    value = _COMPOUND_SUFFIX_RE.sub(r" \1 ", value)

    value = re.sub(r"[-_.]+", " ", value)

    value = _strip_noise(value)
    return _title(value)


def _strip_noise(value: str) -> str:
    """
    Phase 7 – Token-level noise removal.

    OPTIMIZATION: replaced the per-pattern `any(pattern.match(upper) for
    pattern in BANK_OR_TECH_PATTERNS)` loop with a single call to
    `_is_bank_or_tech`, which checks all 12 technical patterns in one
    combined-regex pass instead of up to 12 separate ones per token.
    """
    tokens = []
    for token in re.findall(r"[A-Za-z][A-Za-z0-9&]*", value):
        upper = token.upper()
        if upper in NOISE_WORDS:
            continue
        if _is_bank_or_tech(upper):
            continue
        if len(token) == 1 and upper not in {"D"}:
            continue
        tokens.append(token)
    return " ".join(tokens).strip()


def _title(value: str) -> str:
    """Title-case the cleaned name. Short all-caps words (≤4 chars) are kept as-is."""
    if not value:
        return UNKNOWN
    words = []
    for word in value.split():
        if word.isupper() and len(word) <= 4:
            words.append(word)
        else:
            words.append(word[:1].upper() + word[1:].lower())
    return " ".join(words)


def _is_good_name(value: str) -> bool:
    """
    Phase 7 – Validates whether a cleaned candidate is good enough to keep
    as a counterparty.

    OPTIMIZATION: uses `_is_bank_or_tech` (single combined regex) instead of
    looping over all 12 BANK_OR_TECH_PATTERNS individually.
    """
    if not value or value == UNKNOWN:
        return False

    if value.startswith(("UNKNOWN_", "UNRESOLVED_", "MASKED_")):
        return True

    if len(value) < 3:
        return False

    upper = value.upper().replace(" ", "")

    if upper in NOISE_WORDS:
        return False

    if _is_bank_or_tech(upper):
        return False

    if _is_reference_like(value):
        return False

    if _is_gibberish(value):
        return False

    return bool(re.search(r"[A-Za-z]", value))


# ═════════════════════════════════════════════════════════════════════════════
# Phase 7 – Utility Helpers
# ═════════════════════════════════════════════════════════════════════════════

_MASKED_MOBILE_RE = re.compile(r"MOBILEX+", re.I)


def _is_masked_mobile(value: str) -> bool:
    """Phase 7 – Detects masked mobile numbers (e.g., MOBILEXXXX)."""
    return bool(_MASKED_MOBILE_RE.search(value))


def _digit_ratio(value: str) -> float:
    """Returns the ratio of digit characters to total characters."""
    if not value:
        return 0.0
    digits = sum(c.isdigit() for c in value)
    return digits / len(value)


def _is_reference_like(value: str) -> bool:
    """Phase 7 – Transaction Reference detection."""
    value = re.sub(r"[^A-Za-z0-9]", "", str(value).strip())
    if not value:
        return False
    has_letter = any(c.isalpha() for c in value)
    has_digit = any(c.isdigit() for c in value)
    if has_letter and has_digit and len(value) >= 10:
        return True
    return False


def _is_gibberish(value: str) -> bool:
    """Phase 7 – Gibberish Filtering."""
    if value.upper() in {"P2A", "P2P", "P2M"}:
        return False
    if not re.search(r"[AEIOUYaeiouy]", value):
        return True
    return False

In [18]:
"""
Phase 10+ : Pattern Detection & Feature Engineering Layer
==========================================================
"""

from __future__ import annotations

import re
from collections import Counter
from dataclasses import dataclass

# ═════════════════════════════════════════════════════════════════════════
# Data contract coming IN from the existing pipeline
# ═════════════════════════════════════════════════════════════════════════

@dataclass(frozen=True)
class EntityRow:
    narration: str
    payment_mode: str
    counterparty: str
    flow: str
    transaction_type: str


@dataclass(frozen=True)
class PatternResult:
    transaction_purpose: str
    merchant_category: str
    lifestyle_category: str
    counterparty_type: str
    digital_payment_flag: bool
    cash_transaction_flag: bool
    business_payment_flag: bool
    financial_institution_flag: bool
    income_flag: bool
    loan_flag: bool
    expense_category: str
    pattern_confidence: float
    pattern_rule: str
    risk_tag: str

    def to_dict(self) -> dict:
        return self.__dict__.copy()


UNKNOWN = "UNKNOWN"

# ═════════════════════════════════════════════════════════════════════════
# 1. INCOME PATTERNS
# ═════════════════════════════════════════════════════════════════════════

INCOME_RULES: tuple[tuple[str, re.Pattern[str]], ...] = (
    ("Salary Credit",        re.compile(r"\bSAL(?:ARY)?\b|\bSALARIES\b|\bPAYROLL\b|\bSAL CR\b", re.I)),
    ("Bonus/Incentive Credit", re.compile(r"\bBONUS\b|\bINCENTIVE\b|\bEX[- ]?GRATIA\b", re.I)),
    ("Interest Income",      re.compile(r"\bINT\.?\s*CREDIT\b|\bINTEREST CREDIT\b|\bSAVINGS INT\b", re.I)),
    ("Dividend Income",      re.compile(r"\bDIVIDEND\b|\bDIV\s*CREDIT\b", re.I)),
    ("Refund/Reversal",      re.compile(r"\bREFUND\b|\bREVERSAL\b|\bCHARGEBACK\b|\bREVERSED\b", re.I)),
    ("Cashback Credit",      re.compile(r"\bCASHBACK\b|\bCASH\s*BACK\b", re.I)),
    ("Rental Income",        re.compile(r"\bRENT RECD\b|\bRENTAL INCOME\b", re.I)),
)

# ═════════════════════════════════════════════════════════════════════════
# 2. LOAN / DEBT PATTERNS
# ═════════════════════════════════════════════════════════════════════════

LOAN_RULES: tuple[tuple[str, re.Pattern[str]], ...] = (
    ("Loan EMI",              re.compile(r"\bEMI\b|\bLOAN\s*EMI\b|\bINSTALL?MENT\b", re.I)),
    ("Loan Repayment",        re.compile(r"\bLOAN\s*REC\b|\bLOAN\s*REPAY(?:MENT)?\b|\bLOAN A/?C\b", re.I)),
    ("Loan Disbursement",     re.compile(r"\bLOAN\s*DISB(?:URSEMENT)?\b|\bDISBURSAL\b", re.I)),
    ("Credit Card Bill Payment", re.compile(r"\bCC\s*(?:BILL|PAYMENT)\b|\bCREDIT CARD\b|\bCARD\s*BILL\b", re.I)),
    ("BNPL/NBFC Repayment",   re.compile(r"\bBAJAJ\s*FINSERV\b|\bBNPL\b|\bLAZYPAY\b|\bSIMPL\b|\bSLICE\b", re.I)),
    ("Overdraft/Penalty",     re.compile(r"\bOD\s*INT\b|\bPENAL(?:TY)?\s*(?:CHG|CHARGE)\b|\bBOUNCE\s*CHG\b", re.I)),
)

# ═════════════════════════════════════════════════════════════════════════
# 3. RECURRING BILLS / FINANCIAL DISCIPLINE
# ═════════════════════════════════════════════════════════════════════════

RECURRING_BILL_RULES: tuple[tuple[str, re.Pattern[str]], ...] = (
    ("Rent Payment",          re.compile(r"\bRENT\b(?!AL INCOME)|\bHOUSE\s*RENT\b", re.I)),
    ("Society Maintenance",   re.compile(r"\bMAINTENANCE\b|\bSOCIETY\b|\bRWA\b", re.I)),
    ("Utility Bill",          re.compile(r"\bELECTRICITY\b|\bBESCOM\b|\bMSEB\b|\bTATA\s*POWER\b|"
                                          r"\bGAS\s*BILL\b|\bWATER\s*BILL\b|\bBSES\b|\bADANI\s*ELEC", re.I)),
    ("Telecom Recharge",      re.compile(r"\bJIO\b|\bAIRTEL\s*RECHARGE\b|\bVI\s*RECHARGE\b|\bBSNL\b|\bRECHARGE\b", re.I)),
    ("Subscription/OTT/SaaS", re.compile(r"\bNETFLIX\b|\bHOTSTAR\b|\bPRIME\s*VIDEO\b|\bSPOTIFY\b|"
                                          r"\bYOUTUBE\s*PREMIUM\b|\bSUBSCRIPTION\b|\bSONY\s*LIV\b", re.I)),
    ("Insurance Premium",     re.compile(r"\bLIC\b|\bPREMIUM\b|\bINSURANCE\b|\bHDFC\s*LIFE\b|\bICICI\s*PRU\b|"
                                          r"\bSTAR\s*HEALTH\b|\bBAJAJ\s*ALLIANZ\b", re.I)),
)

# ═════════════════════════════════════════════════════════════════════════
# 4. INVESTMENT PATTERNS
# ═════════════════════════════════════════════════════════════════════════

INVESTMENT_RULES: tuple[tuple[str, re.Pattern[str]], ...] = (
    ("SIP/Mutual Fund",  re.compile(r"\bSIP\b|\bMUTUAL\s*FUND\b|\bICICI\s*PRU(?!DENTIAL LIFE)\b|\bHDFC\s*MF\b|\bAMC\b", re.I)),
    ("Direct Equity/Trading", re.compile(r"\bZERODHA\b|\bGROWW\b|\bUPSTOX\b|\bANGEL\s*ONE\b|\bICICI\s*DIRECT\b|\bDEMAT\b", re.I)),
    ("Fixed Deposit",    re.compile(r"\bFD\s*(?:BOOKING|CREATION)\b|\bFIXED\s*DEPOSIT\b|\bRD\s*INSTAL\b", re.I)),
    ("PF/NPS Contribution", re.compile(r"\bNPS\b|\bPROVIDENT\s*FUND\b|\bEPFO\b|\bPPF\b", re.I)),
)

# ═════════════════════════════════════════════════════════════════════════
# 5. LIFESTYLE / DISCRETIONARY SPEND
# ═════════════════════════════════════════════════════════════════════════

LIFESTYLE_RULES: tuple[tuple[str, str, re.Pattern[str]], ...] = (
    ("Food Delivery",   "Food & Dining", re.compile(r"\bZOMATO\b|\bSWIGGY\b|\bEATERNITY\b", re.I)),
    ("E-commerce Purchase", "Shopping", re.compile(r"\bAMAZON\b|\bFLIPKART\b|\bMYNTRA\b|\bAJIO\b|\bNYKAA\b|\bMEESHO\b", re.I)),
    ("Fuel Purchase",   "Travel/Transport", re.compile(r"\bPETROL\b|\bFUEL\b|\bHPCL\b|\bBPCL\b|\bIOCL\b|\bINDIAN\s*OIL\b", re.I)),
    ("Ride-hailing",    "Travel/Transport", re.compile(r"\bUBER\b|\bOLA\b|\bRAPIDO\b", re.I)),
    ("Travel Booking",  "Travel/Transport", re.compile(r"\bIRCTC\b|\bMAKEMYTRIP\b|\bGOIBIBO\b|\bYATRA\b|\bINDIGO\b|\bAIR\s*INDIA\b|\bREDBUS\b", re.I)),
    ("Gaming/Betting",  "Entertainment", re.compile(r"\bDREAM11\b|\bMPL\b|\bRUMMY\b|\bBET\w*\b|\b1XBET\b|\bPOKER\b", re.I)),
    ("Healthcare",      "Healthcare", re.compile(r"\bHOSPITAL\b|\bAPOLLO\b|\bPHARMEASY\b|\b1MG\b|\bPHARMACY\b|\bMEDICAL\b|\bDIAGNOSTIC\b", re.I)),
    ("Education/School Fee", "Education", re.compile(r"\bSCHOOL\s*FEE\b|\bTUITION\b|\bCOLLEGE\s*FEE\b|\bBYJU\b|\bUNACADEMY\b", re.I)),
    ("Grocery",         "Essentials", re.compile(r"\bBIGBASKET\b|\bBLINKIT\b|\bZEPTO\b|\bDMART\b|\bGROCER\w*\b", re.I)),
)

# ═════════════════════════════════════════════════════════════════════════
# 6. GOVERNMENT / STATUTORY
# ═════════════════════════════════════════════════════════════════════════

GOVERNMENT_RULES: tuple[tuple[str, re.Pattern[str]], ...] = (
    ("Tax Payment",       re.compile(r"\bGST\b|\bINCOME\s*TAX\b|\bTDS\b|\bCHALLAN\b", re.I)),
    ("Government Payment", re.compile(r"\bPMJDY\b|\bPMAY\b|\bDBT\b|\bSUBSIDY\b", re.I)),
    ("Toll/FASTag",       re.compile(r"\bFASTAG\b|\bTOLL\b|\bNETC\b", re.I)),
)

# OPTIMIZATION: this GOVT/GOVERNMENT/MUNICIPAL check was previously an inline
# re.search() literal inside _counterparty_type, recompiled on every call.
# Promoted to a module-level constant like the other rule groups.
_GOVT_RE = re.compile(r"\bGOVT\b|\bGOVERNMENT\b|\bMUNICIPAL\b", re.I)

# ═════════════════════════════════════════════════════════════════════════
# 7. CASH & TRANSFER PATTERNS
# ═════════════════════════════════════════════════════════════════════════

CASH_RULES = re.compile(r"\bATM\b|\bCASH\s*WDL\b|\bCASH\s*WITHDRAWAL\b|\bNFS\b|\bATW\b", re.I)
CASH_DEPOSIT_RULE = re.compile(r"\bCASH\s*DEP(?:OSIT)?\b|\bCDM\b", re.I)
WALLET_LOAD_RULE = re.compile(r"\bWALLET\s*(?:LOAD|TOPUP|TOP-UP)\b|\bADD\s*MONEY\b|\bMOBIKWIK\b|\bFREECHARGE\b", re.I)
SELF_TRANSFER_RULE = re.compile(r"\bSELF\b|\bOWN\s*A/?C\b|\bINTERNAL\s*TRANSFER\b", re.I)
QR_PAYMENT_RULE = re.compile(r"PAYTMQR|UNRESOLVED_QR_MERCHANT|\bQR\b", re.I)
COLLECT_REQUEST_RULE = re.compile(r"\bCOLLECT\b|\bUPI\s*COLLECT\b", re.I)

# ═════════════════════════════════════════════════════════════════════════
# 8. BUSINESS / VENDOR PAYMENT
# ═════════════════════════════════════════════════════════════════════════

BUSINESS_KEYWORDS_RE = re.compile(
    r"\bINVOICE\b|\bVENDOR\b|\bSUPPLIER\b|\bPURCHASE\s*ORDER\b|\bGSTIN\b|"
    r"\bPROPRIETOR\b|\bFIRM\b|\bTRADERS\b|\bENTERPRISES\b|\bWHOLESALE\b",
    re.I,
)

FINANCIAL_INSTITUTION_RE = re.compile(
    r"\bBANK\b|\bNBFC\b|\bFINANCE\b|\bFINSERV\b|\bMUTUAL\s*FUND\b|\bINSURANCE\b|\bAMC\b",
    re.I,
)

# ═════════════════════════════════════════════════════════════════════════
# Counterparty type inference
#
# OPTIMIZATION: business_flag and financial_institution_flag are now optional
# parameters. detect_pattern() already computes both of these once per row
# for its own use (business_payment_flag / financial_institution_flag output
# fields) — previously _counterparty_type() silently recomputed both via its
# own re.search() calls, meaning BUSINESS_KEYWORDS_RE and
# FINANCIAL_INSTITUTION_RE were each evaluated TWICE per row. Passing the
# already-computed flags in eliminates that duplicate regex work. Calling
# _counterparty_type() without the optional args still works exactly as
# before (falls back to computing them), so this is backward compatible.
# ═════════════════════════════════════════════════════════════════════════

def _counterparty_type(
    counterparty: str,
    narration: str,
    business_flag: bool | None = None,
    financial_institution_flag: bool | None = None,
) -> str:
    if counterparty.startswith("MASKED_COUNTERPARTY") or "Individual" in counterparty:
        return "Individual"
    if "Merchant" in counterparty or "UNKNOWN_MERCHANT" in counterparty:
        return "Merchant"
    is_financial_institution = (
        financial_institution_flag
        if financial_institution_flag is not None
        else bool(FINANCIAL_INSTITUTION_RE.search(narration))
    )
    if is_financial_institution:
        return "Financial Institution"
    if _GOVT_RE.search(narration):
        return "Government"
    is_business = business_flag if business_flag is not None else bool(BUSINESS_KEYWORDS_RE.search(narration))
    if is_business:
        return "Business/Vendor"
    if counterparty in ("TRANSACTION_REFERENCE", "UNRESOLVED_QR_MERCHANT", UNKNOWN):
        return "Unresolved"
    return "Merchant"


# ═════════════════════════════════════════════════════════════════════════
# Core dispatcher
# ═════════════════════════════════════════════════════════════════════════

def detect_pattern(row: EntityRow) -> PatternResult:
    text = row.narration.upper()
    is_credit = row.transaction_type.strip().upper() == "CREDIT"
    is_debit = row.transaction_type.strip().upper() == "DEBIT"

    # OPTIMIZATION: these four regex checks were each being run TWICE in the
    # original code — once inside the dispatch chain (steps 7–9) and again
    # afterwards to compute the standalone flag fields (cash_flag,
    # financial_institution_flag, business_flag) and counterparty_type.
    # Computing them once up front and reusing the booleans below removes
    # 4 duplicate regex searches per row.
    business_flag = bool(BUSINESS_KEYWORDS_RE.search(text))
    financial_institution_flag = bool(FINANCIAL_INSTITUTION_RE.search(text))
    cash_withdrawal_hit = bool(CASH_RULES.search(text))
    cash_deposit_hit = bool(CASH_DEPOSIT_RULE.search(text))
    counterparty_type = _counterparty_type(row.counterparty, text, business_flag, financial_institution_flag)

    purpose = UNKNOWN
    merchant_category = UNKNOWN
    lifestyle_category = "N/A"
    confidence = 0.0
    rule = "no_pattern"
    income_flag = False
    loan_flag = False
    expense_category = UNKNOWN

    # ── 1. Income (only meaningful on CREDIT legs) ──────────────────────
    if is_credit:
        for label, pattern in INCOME_RULES:
            if pattern.search(text):
                purpose, confidence, rule = label, 0.85, "income_rule"
                income_flag = True
                merchant_category = "Income"
                break

    # ── 2. Loan / debt ───────────────────────────────────────────────────
    if purpose == UNKNOWN:
        for label, pattern in LOAN_RULES:
            if pattern.search(text):
                purpose, confidence, rule = label, 0.85, "loan_rule"
                loan_flag = True
                merchant_category = "Debt Obligation"
                expense_category = "Debt Repayment"
                break

    # ── 3. Recurring bills / discipline ─────────────────────────────────
    if purpose == UNKNOWN:
        for label, pattern in RECURRING_BILL_RULES:
            if pattern.search(text):
                purpose, confidence, rule = label, 0.80, "recurring_bill_rule"
                merchant_category = "Recurring Obligation"
                expense_category = "Fixed Expense"
                break

    # ── 4. Investment ────────────────────────────────────────────────────
    if purpose == UNKNOWN:
        for label, pattern in INVESTMENT_RULES:
            if pattern.search(text):
                purpose, confidence, rule = label, 0.80, "investment_rule"
                merchant_category = "Investment"
                expense_category = "Savings/Investment"
                break

    # ── 5. Government / statutory ───────────────────────────────────────
    if purpose == UNKNOWN:
        for label, pattern in GOVERNMENT_RULES:
            if pattern.search(text):
                purpose, confidence, rule = label, 0.75, "government_rule"
                merchant_category = "Government/Statutory"
                expense_category = "Statutory Payment"
                break

    # ── 6. Lifestyle / discretionary ────────────────────────────────────
    if purpose == UNKNOWN:
        for label, category, pattern in LIFESTYLE_RULES:
            if pattern.search(text):
                purpose, confidence, rule = label, 0.75, "lifestyle_rule"
                merchant_category = category
                lifestyle_category = category
                expense_category = "Discretionary Spend" if category in (
                    "Food & Dining", "Shopping", "Entertainment") else "Essential Spend"
                break

    # ── 7. Cash & transfer mechanics ────────────────────────────────────
    # OPTIMIZATION: uses the pre-computed cash_withdrawal_hit / cash_deposit_hit
    # booleans instead of re-running CASH_RULES / CASH_DEPOSIT_RULE again.
    if purpose == UNKNOWN:
        if cash_withdrawal_hit:
            purpose, confidence, rule = "Cash Withdrawal", 0.85, "cash_atm_rule"
            merchant_category = "Cash"
            expense_category = "Cash Withdrawal"
        elif cash_deposit_hit:
            purpose, confidence, rule = "Cash Deposit", 0.85, "cash_deposit_rule"
            merchant_category = "Cash"
            income_flag = is_credit
        elif WALLET_LOAD_RULE.search(text):
            purpose, confidence, rule = "Wallet Loading", 0.75, "wallet_load_rule"
            merchant_category = "Wallet"
        elif QR_PAYMENT_RULE.search(text) or row.counterparty == "UNRESOLVED_QR_MERCHANT":
            purpose, confidence, rule = "QR Payment", 0.60, "qr_payment_rule"
            merchant_category = "Merchant Payment"
        elif COLLECT_REQUEST_RULE.search(text):
            purpose, confidence, rule = "UPI Collect Request", 0.70, "collect_request_rule"
            merchant_category = "P2P/Collect"
        elif SELF_TRANSFER_RULE.search(text):
            purpose, confidence, rule = "Internal Account Transfer", 0.75, "self_transfer_rule"
            merchant_category = "Internal Transfer"

    # ── 8. Business / vendor payment ────────────────────────────────────
    # OPTIMIZATION: reuses business_flag computed at the top of the function
    # instead of re-running BUSINESS_KEYWORDS_RE.search(text) again.
    if purpose == UNKNOWN and business_flag:
        purpose, confidence, rule = "Business Payment", 0.65, "business_vendor_rule"
        merchant_category = "Business/Vendor"

    # ── 9. Fallback: generic P2P / merchant based on entity role ────────
    # OPTIMIZATION: reuses counterparty_type computed once at the top instead
    # of calling _counterparty_type() a second time.
    if purpose == UNKNOWN:
        if counterparty_type == "Individual":
            purpose, confidence, rule = "P2P Transfer", 0.45, "fallback_p2p"
            merchant_category = "P2P"
        elif counterparty_type in ("Merchant", "Business/Vendor"):
            purpose, confidence, rule = "Merchant Payment", 0.40, "fallback_merchant"
            merchant_category = "Merchant Payment"
        else:
            purpose, confidence, rule = UNKNOWN, 0.0, "no_pattern"

    digital_flag = row.payment_mode not in ("Cash", "ATM", UNKNOWN)
    cash_flag = cash_withdrawal_hit or cash_deposit_hit

    if is_debit and expense_category == UNKNOWN and purpose not in (UNKNOWN,):
        expense_category = "Other Expense"

    # ── Risk tagging ──────────────────────────────────────────────────
    risk_tag = "None"
    if row.counterparty in ("UNRESOLVED_QR_MERCHANT", "TRANSACTION_REFERENCE"):
        risk_tag = "Low Traceability"
    if purpose == "Gaming/Betting":
        risk_tag = "Discretionary Risk - Gambling"
    if purpose == "P2P Transfer" and counterparty_type == "Individual" and row.payment_mode == "UPI":
        risk_tag = "Monitor - High-Frequency P2P (check velocity downstream)"

    return PatternResult(
        transaction_purpose=purpose,
        merchant_category=merchant_category,
        lifestyle_category=lifestyle_category,
        counterparty_type=counterparty_type,
        digital_payment_flag=digital_flag,
        cash_transaction_flag=cash_flag,
        business_payment_flag=business_flag,
        financial_institution_flag=financial_institution_flag,
        income_flag=income_flag,
        loan_flag=loan_flag,
        expense_category=expense_category,
        pattern_confidence=confidence,
        pattern_rule=rule,
        risk_tag=risk_tag,
    )


# ═════════════════════════════════════════════════════════════════════════
# DataFrame-level feature engineering
# ═════════════════════════════════════════════════════════════════════════

def add_recurring_flag(df, counterparty_col="COUNTERPARTY", min_occurrences=3):
    """Marks a counterparty as RECURRING if it appears >= min_occurrences times."""
    counts = Counter(df[counterparty_col])
    df = df.copy()
    df["RECURRING_FLAG"] = df[counterparty_col].map(
        lambda c: counts[c] >= min_occurrences and c not in (
            UNKNOWN, "TRANSACTION_REFERENCE", "UNRESOLVED_QR_MERCHANT"
        )
    )
    return df


def salary_stability_score(df, purpose_col="TRANSACTION_PURPOSE"):
    """Rough proxy: count of distinct months with a 'Salary Credit' pattern."""
    salary_hits = (df[purpose_col] == "Salary Credit").sum()
    return {
        "salary_credit_count": int(salary_hits),
        "has_regular_salary": bool(salary_hits >= 3),
    }


def discretionary_spend_ratio(df, expense_col="EXPENSE_CATEGORY", debit_mask=None):
    """Ratio of discretionary-labelled debits to all classified debits."""
    debits = df if debit_mask is None else df[debit_mask]
    total = (debits[expense_col] != UNKNOWN).sum()
    discretionary = (debits[expense_col] == "Discretionary Spend").sum()
    return round(discretionary / total, 3) if total else 0.0



def apply_pattern_layer(df):
    """
    Expects df already has: NARRATION, TYPE, PAYMENT_APP, COUNTERPARTY,
    ENTITY_ROLE from the existing pipeline.

    OPTIMIZATION: previously did `.map(detect_pattern).map(lambda p: p.to_dict())`
    — two full passes over the Series. Combined into a single `.map()` call
    below, so each row is only visited once instead of twice.
    """
    rows = df.apply(
        lambda r: EntityRow(
            narration=str(r.get("NARRATION", "")),
            payment_mode=str(r.get("PAYMENT_APP", UNKNOWN)),
            counterparty=str(r.get("COUNTERPARTY", UNKNOWN)),
            flow=str(r.get("ENTITY_ROLE", UNKNOWN)),
            transaction_type=str(r.get("TYPE", "")),
        ),
        axis=1,
    )
    results = rows.map(lambda row: detect_pattern(row).to_dict())
    pattern_df = type(df)(results.tolist())

    for col in pattern_df.columns:
        df[col.upper()] = pattern_df[col].values

    df = add_recurring_flag(df)
    return df

In [19]:
# ── Constants to keep ENTITY_ROLE labels in sync everywhere they're used ───
PAYER_LABEL = "Payer (Source)"
PAYEE_LABEL = "Payee (Recipient)"

# ── Reusable bank-leak audit check (replaces the two duplicate commented blocks) ──
def check_bank_leak(df, extra_terms=r"|Escrow|Payout"):
    """Flags COUNTERPARTY values that still contain bank/corporate noise."""
    pattern = r'Bank|Ltd|Pvt|Private|Limited|Fintech' + extra_terms
    mask = df['COUNTERPARTY'].str.contains(pattern, case=False, na=False)
    return df[mask]['COUNTERPARTY'].value_counts()

# Run only when auditing extraction quality, not on every pipeline pass:
# print(check_bank_leak(df).head(50))

# ── Phase 1: Entity extraction ──────────────────────────────────────────────
parsed = df.apply(lambda row: parse_row(row).to_dict(), axis=1)
parsed_df = pd.DataFrame(parsed.tolist())

df["PAYMENT_APP"] = parsed_df["payment_mode"]
df["COUNTERPARTY"] = parsed_df["counterparty"]
df["ENTITY_ROLE"] = parsed_df["flow"].map({
    "Sender": PAYER_LABEL,
    "Receiver": PAYEE_LABEL,
}).fillna("Unknown Role")
df["EXTRACTION_CONFIDENCE"] = parsed_df["confidence"]
df["EXTRACTION_RULE"] = parsed_df["rule"]

# Assigned so it can be inspected/exported later, not just printed once
payment_app_counterparty_counts = df[['PAYMENT_APP', 'COUNTERPARTY']].value_counts().reset_index()
print(payment_app_counterparty_counts)

# ── Phase 2: Pattern detection layer ────────────────────────────────────────
# Make sure df has a TYPE column matching CREDIT/DEBIT
# (rename if yours differs, e.g. df["TYPE"] = df["TRANSACTION_TYPE"])
df = apply_pattern_layer(df)

# ── Diagnostics: gathered once so runs can be compared across iterations ───
diagnostics = {
    "payment_app_x_counterparty": payment_app_counterparty_counts,
    "transaction_purpose_counts": df["TRANSACTION_PURPOSE"].value_counts(),
    "pattern_rule_coverage": df["PATTERN_RULE"].value_counts(),
    "salary_stability": salary_stability_score(df),
    "discretionary_spend_ratio": discretionary_spend_ratio(
        df, debit_mask=(df["ENTITY_ROLE"] == PAYEE_LABEL)
    ),
}

print("Transaction Purpose distribution:\n", diagnostics["transaction_purpose_counts"], "\n")
print("Pattern rule coverage:\n", diagnostics["pattern_rule_coverage"], "\n")
print("Salary stability:", diagnostics["salary_stability"])
print("Discretionary spend ratio:", diagnostics["discretionary_spend_ratio"])

# Spot-check sample (kept separate since it's a fresh random sample each run)
df[["NARRATION", "PAYMENT_APP", "COUNTERPARTY", "TRANSACTION_PURPOSE", "MERCHANT_CATEGORY", "RISK_TAG"]].sample(30)

      PAYMENT_APP                     COUNTERPARTY  count
0             UPI              MASKED_COUNTERPARTY  50109
1             UPI                          UNKNOWN  31628
2         PhonePe   MASKED_COUNTERPARTY - YES Bank  15448
3         PhonePe  MASKED_COUNTERPARTY - Axis Bank   9474
4           Paytm                          UNKNOWN   6653
...           ...                              ...    ...
42068         UPI                     Harshitsingh      1
42069         UPI                    Harshitsaxena      1
42070         UPI                  Harshitkumarsin      1
42071         UPI                     Harshitha KR      1
42072         UPI                     Harshitha DO      1

[42073 rows x 3 columns]
Transaction Purpose distribution:
 TRANSACTION_PURPOSE
Merchant Payment             206380
P2P Transfer                  88818
UNKNOWN                       65139
QR Payment                    12002
Cash Withdrawal                3443
Telecom Recharge               2068
Subscri

,NARRATION,PAYMENT_APP,COUNTERPARTY,TRANSACTION_PURPOSE,MERCHANT_CATEGORY,RISK_TAG
36768,UPI/CR/332839369637/SRI VENKA/KKBK/**kotak@ibl...,UPI,SRI Venka,Merchant Payment,Merchant Payment,None
298033,UPI-81491900048116-MOBILEXXXX-2@YBL-2776084258...,Google Pay,UNKNOWN,UNKNOWN,UNKNOWN,None
45573,UPI/507320200599/235547/UPI/nileshchorpgarokhd,UPI,Nileshchorpgarokhd,Merchant Payment,Merchant Payment,None
162017,UPI/DR/254980441647/Sajpe/YESB/002261100000025...,PhonePe,Trimurti Coldrinks AND General Store,Merchant Payment,Merchant Payment,None
312760,UPI/Ramchandra/564705570814/UPI,UPI,Rakesh Ashokbha,Merchant Payment,Merchant Payment,None
79599,UPI/430623509982/202332/UPI/gkparmar449okhdfcb,UPI,MASKED_COUNTERPARTY,P2P Transfer,P2P,Monitor - High-Frequency P2P (check velocity d...
121344,UPI-ANIL SINGH-ANILSINGHS22061@IBL-HDFC0002285...,Paytm,MASKED_COUNTERPARTY - @pthdfc,P2P Transfer,P2P,None
246498,UPI/120469876656/184716/UPI/MOBILEXXXXybl/Paym,UPI,MASKED_COUNTERPARTY,P2P Transfer,P2P,Monitor - High-Frequency P2P (check velocity d...
329058,UPI/410668598393/CR/Jagd/AIRP/MOBILEXXXX@ibl/P...,UPI,MASKED_COUNTERPARTY - IndusInd Bank,P2P Transfer,P2P,Monitor - High-Frequency P2P (check velocity d...
70349,UPI-KARTHIK WINES-PAYTMQR5EHJ5A@PTYS-YESB0PTMU...,IMPS,UNKNOWN,QR Payment,Merchant Payment,None
